In [8]:
import tarfile
import sagemaker

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.onnx", arcname="model.onnx")

session = sagemaker.Session()
model_data = session.upload_data(
    "model.tar.gz",
    bucket="sagemaker-eu-north-1-527557823764",
    key_prefix=f"smart-parking-lot/endpoint-models/v{mv.version}",
)
print(model_data)

s3://sagemaker-eu-north-1-527557823764/smart-parking-lot/endpoint-models/v1/model.tar.gz


In [9]:
from sagemaker import get_execution_role
from sagemaker.sklearn import SKLearnModel
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

ROLE_ARN = get_execution_role()
ENDPOINT_NAME = f"parking-occupancy-onnx-v{mv.version}"

endpoint_model = SKLearnModel(
    model_data=model_data,
    role=ROLE_ARN,
    entry_point="inference.py",
    source_dir="/home/sagemaker-user/smart-parking-lot/serving",
    framework_version="1.2-1",
    py_version="py3",
)

predictor = endpoint_model.deploy(
    endpoint_name=ENDPOINT_NAME,
    serverless_inference_config=ServerlessInferenceConfig(
        memory_size_in_mb=2048,
        max_concurrency=2,
    ),
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
)
print("Endpoint:", ENDPOINT_NAME)

/home/sagemaker-user/deploy-env/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.
INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-10-25-15-27-50-763
INFO:sagemaker:Creating endpoint-config with name parking-occupancy-onnx-v1
INFO:sagemaker:Creating endpoint with name parking-occupancy-onnx-v1


----*

ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:19                                                                                   │
│                                                                                                  │
│   16 │   py_version="py3",                                                                       │
│   17 )                                                                                           │
│   18                                                                                             │
│ ❱ 19 predictor = endpoint_model.deploy(                                                          │
│   20 │   endpoint_name=ENDPOINT_NAME,                                                            │
│   21 │   serverless_inference_config=ServerlessInferenceConfig(                                  │
│   22 │   │   memory_size_in_mb=2048,                                                             │
│                                                                                                  │
│ /home/sagemaker-user/deploy-env/lib/python3.12/site-packages/sagemaker/model.py:1837 in deploy   │
│                                                                                                  │
│   1834 │   │   │   │   )                                                                         │
│   1835 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1836 │   │   │   else:                                                                         │
│ ❱ 1837 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1838 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1839 │   │   │   │   │   production_variants=[production_variant],                             │
│   1840 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ /home/sagemaker-user/deploy-env/lib/python3.12/site-packages/sagemaker/session.py:6431 in        │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6428 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6429 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6430 │   │                                                                                     │
│ ❱ 6431 │   │   return self.create_endpoint(                                                      │
│   6432 │   │   │   endpoint_name=name,                                                           │
│   6433 │   │   │   config_name=name,                                                             │
│   6434 │   │   │   tags=endpoint_tags,                                                           │
│                                                                                                  │
│ /home/sagemaker-user/deploy-env/lib/python3.12/site-packages/sagemaker/session.py:5259 in        │
│ create_endpoint                                                                                  │
│                                                                                                  │
│   5256 │   │   │   logger.error(                                                                 │
│   5257 │   │   │   │   "Please check the troubleshooting guide for common errors: %s", troubles  │
│   5258 │   │   │   )                                                                             │
│ ❱ 5259 │   │   │   raise e                                                                       │
│   5260 │                                                   

In [ ]:
row = sample.values.tolist()   # [[400, 9, 1, 11, 0, -9.148393, 38.731375]]
print(predictor.predict(row))